# FCD SynthSeg — 7-Class FCD Lesion Segmentation

**Task:** Focal Cortical Dysplasia (FCD Type II) lesion segmentation from 3-D FLAIR MRI.  
**Approach:** SynthSeg-style training — synthetic images are generated on-the-fly from label maps
using a per-class Gaussian Mixture Model (one (μ, σ) range per tissue class). Subject-specific
FCD appearance augmentations are injected into the synthetic branch. The network trains
exclusively on synthetic images and is evaluated on real FLAIR scans.

**7 output classes:**

| Class | Tissue |
|-------|--------|
| 0 | Background |
| 1 | White Matter |
| 2 | Cerebral Cortex |
| 3 | Deep Gray Matter |
| 4 | CSF |
| 5 | WM-GM Separator (label 18) |
| 6 | FCD Lesion |

**End-to-end pipeline:**
```
labelmap.nii  →  one-hot (19, D, H, W)
              →  PerClassGMM           →  synthetic FLAIR  [~0, 255]
              →  FCDAugmentations      →  FCD appearance injected
              →  IntensityTransform    →  normalized        [0, 1]
              →  label fusion          →  ROI voxels → class 6
              →  SegNet forward
              →  DiceCE loss  →  backward (synthetic branch only)
Real FLAIR passes through the network forward-only for monitoring (no gradient).
```

---
## 1. Run Configuration

Set the experiment name and training duration here before launching.  
All other hyperparameters are passed as CLI flags in §7.

In [1]:
# ──────────────────── Training Configuration ────────────────────
TRAINING_TIME_MINUTES = (11 * 60) + 45                          # 11 hours and 45 mintues

# ──────────────────── Experiment Setup ──────────────────────────
RUN_NAME              = "fcd_experiment_run"                    # Outputs saved to: experiments/<RUN_NAME>/

# ──────────────────── Data & Checkpoints ────────────────────────
INPUT                 = ""                                      # Complete dataset path before /experiment


# ──────────────────── Git & Base Paths ────────────────────────
REPO_URL              = "https://github.com/YassienTawfikk/SynthFCD.git"
BRANCH_NAME           = ""

# ──────────────────── Git & Base Paths ────────────────────────
CLEAR_IMAGES_ON_START = True                                    # Set this flag to True to wipe the images directory (To reduce kaggle/working storage)

---
## 2. Environment Setup

Pinned dependencies to guarantee reproducibility on Kaggle P100/T4 instances.

| Package | Reason |
|---------|--------|
| `cornucopia` | SynthSeg synthesis transforms — pinned commit for stability |
| `torch==2.6.0+cu124` | CUDA 12.4 alignment; fixes torchvision NMS issues |
| `numpy<2.0` | Avoids `_ARRAY_FUNCTION_DISPATCH` ABI breakage with older extensions |
| `torchmetrics` | `DiceScore` metric for validation |
| `pytorch-lightning` | Training loop, checkpointing, CLI |

> ⚠️ **Do not upgrade `cornucopia` past the pinned commit** — the synthesis API is not stable.

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT'] = '0'
os.environ['PYTHONFROZENMODULES'] = 'off'

In [ ]:
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

# ── 1. Cornucopia (pinned commit for stability) ──────────────────────────────
!pip install -qq git+https://github.com/balbasty/cornucopia@6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b > /dev/null 2>&1

# ── 2. Core dependencies ─────────────────────────────────────────────────────
!pip install -qq -U        "jsonargparse[signatures]>=4.27.7" > /dev/null 2>&1
!pip install -qq --no-deps "protobuf==3.20.3"                 > /dev/null 2>&1



# ── 3. PyTorch stack — GPU P100 (CUDA 12.4) ──────────────────────────────────
!pip install -qq "torch==2.6.0+cu124" "torchvision==0.21.0+cu124" "torchaudio==2.6.0+cu124" \
               --index-url https://download.pytorch.org/whl/cu124 > /dev/null 2>&1

# ── 4. NumPy < 2.0  (fixes _ARRAY_API compatibility issues) ──────────────────
!pip install -qq "numpy<2.0" > /dev/null 2>&1

# ── 5. Scientific + visualization stack ──────────────────────────────────────
!pip install -qq pandas matplotlib scipy monai > /dev/null 2>&1

print("All dependencies installed!")

---
## 3. Package & Directory Setup

Copies the `learn2synth` package from the Kaggle input dataset into `/kaggle/working/`
so it is importable as a regular Python package, and creates the `scripts/` folder
where the training script will be written.

In [ ]:
import shutil
import csv
import subprocess
import datetime




WORKING_DIR = "/kaggle/working"
CLONED_REPO_DIR = os.path.join(WORKING_DIR, "SynthFCD")

PACKAGE_DIR = os.path.join(WORKING_DIR, "learn2synth")
STATS_DIR   = os.path.join(WORKING_DIR, "flair-stats-synthseg")
SCRIPTS_DIR = os.path.join(WORKING_DIR, "scripts")

os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# ============================================================
# Choose source: clone repo if INPUT is None, else use INPUT
# ============================================================
if not os.path.exists(CLONED_REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL, CLONED_REPO_DIR], check=True, capture_output=True)
else:
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "fetch", "origin"], check=True, capture_output=True)
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "checkout", BRANCH_NAME], check=True, capture_output=True)
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "pull", "origin", BRANCH_NAME], check=True, capture_output=True)

SOURCE_ROOT = CLONED_REPO_DIR


# ============================================================
# Copy learn2synth Directory
# ============================================================
INPUT_SOURCE = os.path.join(SOURCE_ROOT, "learn2synth")

if os.path.exists(PACKAGE_DIR):
    shutil.rmtree(PACKAGE_DIR)

if os.path.exists(INPUT_SOURCE):
    shutil.copytree(INPUT_SOURCE, PACKAGE_DIR)
else:
    raise FileNotFoundError(f"learn2synth not found at: {INPUT_SOURCE}")


# ============================================================
# Copy flair_stats & scripts Contents
# ============================================================
possible_stats_sources = [
    os.path.join(SOURCE_ROOT, "flair_stats"),
    os.path.join(SOURCE_ROOT, "scripts"),
]

for source_dir in possible_stats_sources:

    if not os.path.exists(source_dir):
        print(f"Directory not found: {source_dir}")
        continue

    # decide destination
    if os.path.basename(source_dir) == "scripts":
        destination_dir = SCRIPTS_DIR
    else:
        destination_dir = STATS_DIR

    for item in os.listdir(source_dir):
        src_path = os.path.join(source_dir, item)
        dst_path = os.path.join(destination_dir, item)

        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
        else:
            shutil.copy2(src_path, dst_path)


# ============================================================
# Remove cloned repo only if we cloned it
# ============================================================
if os.path.exists(CLONED_REPO_DIR):
    shutil.rmtree(CLONED_REPO_DIR)

print("Library setup complete.")

---
## 4. Pre-Launch Setup

Prepares the experiment directory, resolves which checkpoint to resume from, and preserves metric history before the training process can overwrite it.

### 4a. Restore Run Directory

Restores the experiment run directory from `INPUT/experiments/<RUN_NAME>` into `/kaggle/working/experiments/`. Handles `CLEAR_IMAGES_ON_START` flag.

In [ ]:
# ============================================================
# Experiment Paths
# ============================================================

EXPERIMENTS_SRC = f"{INPUT}/experiments"
EXPERIMENTS_DIR = os.path.join(WORKING_DIR, "experiments")

SRC_RUN_DIR = os.path.join(EXPERIMENTS_SRC, RUN_NAME)
RUN_DIR     = os.path.join(EXPERIMENTS_DIR, RUN_NAME)


# ============================================================
# Restore or Initialize Run Directory
# ============================================================

if os.path.exists(SRC_RUN_DIR):
    shutil.copytree(SRC_RUN_DIR, RUN_DIR, dirs_exist_ok=True)

else:
    os.makedirs(RUN_DIR, exist_ok=True)

# ============================================================
# Clear Previous Images (Conditional)
# ============================================================
images_dir = os.path.join(RUN_DIR, "images")

if CLEAR_IMAGES_ON_START and os.path.exists(images_dir):
    shutil.rmtree(images_dir)
    print(f"Flag Active: Cleared previous images → {images_dir}")
elif not CLEAR_IMAGES_ON_START:
    print(f"Flag Inactive: Preserving existing images in → {images_dir}")
else:
    print(f"No previous images directory found (fresh run).")

os.makedirs(images_dir, exist_ok=True)


### 4b. Checkpoint Resolver

Resolves which checkpoint to resume from. Priority:
1. `resume.ckpt` — written every epoch by `EveryEpochCheckpointCallback`
2. `last.ckpt` — `ModelCheckpoint` fallback
3. Fresh start if neither exists

In [ ]:
# ============================================================
# Checkpoint Paths
# ============================================================

CKPT_DIR = os.path.join(SRC_RUN_DIR, "checkpoints")

LAST_CKPT   = os.path.join(CKPT_DIR, "last.ckpt")


# ============================================================
# Select Checkpoint
# ============================================================

if os.path.exists(LAST_CKPT):
    CKPT_PATH   = LAST_CKPT
    ckpt_source = "last.ckpt — ModelCheckpoint fallback"

else:
    CKPT_PATH   = None
    ckpt_source = "fresh start"


# ============================================================
# Summary
# ============================================================

print(f"Checkpoint : {ckpt_source}")

if CKPT_PATH:
    print(f"Path       : {CKPT_PATH}")

### 4c. Preserve Metric History

Merges `metrics.csv` into `metrics_history.csv` before the CSV logger can restart or overwrite it on resume. Must run before the launch cell.

#### 4c.1 Metric CSV helpers

In [ ]:
def _is_value_present(value):
    """Return True if a CSV cell has a real value."""
    if value is None:
        return False
    return str(value).strip() not in ("", "nan", "NaN", "None")


def _safe_number(value, default=float("inf")):
    """Convert CSV string values to numbers for stable sorting."""
    try:
        return float(value) if _is_value_present(value) else default
    except Exception:
        return default

#### 4c.2 CSV read / write


In [ ]:
def read_metric_csv(csv_path):
    """Read a Lightning metrics CSV using only the standard library."""
    if not os.path.exists(csv_path):
        return [], []
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames or [])
    return fieldnames, rows


def write_metric_csv(csv_path, rows, fieldnames):
    """Write merged metric rows using a stable union of all CSV columns."""
    if not rows:
        return
    all_columns = list(fieldnames)
    for row in rows:
        for col in row:
            if col not in all_columns:
                all_columns.append(col)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_columns)
        writer.writeheader()
        writer.writerows(rows)

#### 4c.3 — Row merging logic

In [ ]:
def combine_metric_rows_for_history(rows):
    """
    Merge duplicate Lightning metric rows while keeping non-empty values.

    Lightning often writes different metrics on separate rows for the same
    epoch/step. This combines those rows safely without pandas.
    """
    if not rows:
        return []
    merged = {}
    for row in rows:
        key = (row.get("epoch", ""), row.get("step", ""))
        if key not in merged:
            merged[key] = dict(row)
        else:
            for col, value in row.items():
                if not _is_value_present(merged[key].get(col)) and _is_value_present(value):
                    merged[key][col] = value
    return sorted(
        merged.values(),
        key=lambda r: (_safe_number(r.get("epoch")), _safe_number(r.get("step"))),
    )

#### 4c.4 — Main function

In [ ]:
def update_metrics_history_before_launch(run_dir, checkpoint_path, make_snapshot=True):
    """Preserve metrics_history.csv before launching resumed training."""
    if not checkpoint_path:
        print("[MetricsHistory] Fresh start — no history preservation needed.")
        return

    if not os.path.isdir(run_dir):
        print(f"[MetricsHistory] Run directory not found: {run_dir}")
        return

    metrics_path = os.path.join(run_dir, "metrics.csv")
    history_path = os.path.join(run_dir, "metrics_history.csv")

    if not os.path.exists(metrics_path):
        print("[MetricsHistory] Checkpoint found, but no metrics.csv exists yet.")
        return

    history_fields, history_rows = read_metric_csv(history_path)
    metrics_fields, metrics_rows = read_metric_csv(metrics_path)

    all_rows = history_rows + metrics_rows

    if not all_rows:
        print("[MetricsHistory] Existing metric files are empty. Nothing to preserve.")
        return

    fieldnames = history_fields + [c for c in metrics_fields if c not in history_fields]
    merged = combine_metric_rows_for_history(all_rows)
    write_metric_csv(history_path, merged, fieldnames)


    if make_snapshot:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        snapshot_path = os.path.join(run_dir, f"metrics_before_resume_{timestamp}.csv")
        shutil.copy2(metrics_path, snapshot_path)

#### 4c.5 — Execution


In [ ]:
update_metrics_history_before_launch(
    run_dir=RUN_DIR,
    checkpoint_path=CKPT_PATH,
    make_snapshot=True,
)

#### 4d. Cleanup Redundant Files

In [ ]:
if os.path.isdir(RUN_DIR):
    removed = 0
    for fname in os.listdir(RUN_DIR):
        if fname.startswith("metrics_before_resume_") and fname.endswith(".csv"):
            os.remove(os.path.join(RUN_DIR, fname))
            removed += 1
        elif fname.startswith("events.out.tfevents."):
            os.remove(os.path.join(RUN_DIR, fname))
            removed += 1

---
## 5. Launch

Key CLI flags:
- `--model.flair_modality true` — required for the FLAIR synthesis path
- `--data.eval 0.2` — holds out 20% of subjects for validation (~11 from 57)
- `--trainer.precision 16-mixed` — AMP enabled; synthesis runs under `autocast(..., enabled=False)`

In [ ]:
ckpt_arg = f'--ckpt_path "{CKPT_PATH}"' if CKPT_PATH else ""

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  L2S_RUN_NAME={RUN_NAME} \
  L2S_TIME_LIMIT_MINUTES={TRAINING_TIME_MINUTES} \
  python /kaggle/working/scripts/train_non_parametric_synthFCD.py fit \
    {ckpt_arg} \
    \
    --data.batch_size 2 \
    --data.num_workers 4 \
    --data.eval 0.2 \
    --data.split_seed 42 \
    --data.use_extra_data true \
    \
    --model.native_synthesis false \
    --model.flair_modality true \
    --model.seg_nb_levels 6 \
    --model.seg_features "[16,32,64,128,256,512]" \
    --model.time_limit_minutes {TRAINING_TIME_MINUTES} \
    --model.val_diagnostics_interval 10 \
    --model.debug_subject_ids '["sub-00001", "sub-00033", "sub-00044", "sub-00002", "sub-00058", "sub-00065"]' \
    \
    --trainer.default_root_dir {EXPERIMENTS_DIR} \
    --trainer.max_epochs 2000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 16-mixed \
    --trainer.enable_progress_bar false \
    --trainer.log_every_n_steps 5 \
    --trainer.num_sanity_val_steps 0 \
    \
    --checkpoint.save_top_k 1 \
    --checkpoint.monitor eval_loss \
    --checkpoint.mode min \
    --checkpoint.save_last true \
    \
    --seed_everything 0

---
## 6. Utilities

### Reset Experiment Directory

Uncomment and run to wipe the run directory for a clean restart.

In [ ]:
# shutil.make_archive('zip_file_title', 'zip', 'directory_path (to be zipped)')

In [ ]:
# [shutil.rmtree(f.path, ignore_errors=True) if f.is_dir() else os.unlink(f.path) for f in os.scandir("/kaggle/working/")]